# Model 2 – Fact Bank and Viability Scores

## 0. Load inputs

In [5]:
import os, re, json, warnings
import numpy as np, pandas as pd
from pathlib import Path
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings("ignore")
rng = np.random.default_rng(42)

REQUIRED = ["monitoring_visits_clean.csv", "market_livelihood_summary.csv", "market_series_stats.csv",
            "market_monthly_clean.csv", "market_disaster_district.csv", "area_index.csv",
            "model1_training.csv", "BRIGHT_clean_all_sheets.xlsx"]
missing = [f for f in REQUIRED if not Path(f).exists()]
assert not missing, f"Upload these files first: {missing}"

OUT = Path("model2_outputs"); OUT.mkdir(exist_ok=True)
vis = pd.read_csv("monitoring_visits_clean.csv")
msum = pd.read_csv("market_livelihood_summary.csv")
mstat = pd.read_csv("market_series_stats.csv")
mmon = pd.read_csv("market_monthly_clean.csv", usecols=["series_key", "month", "value", "category", "evidence_type"])
dis = pd.read_csv("market_disaster_district.csv")
area1 = pd.read_csv("area_index.csv")
people = pd.read_csv("model1_training.csv")

CATS = ["AGRICULTURE & PLANTATION CROPS", "LIVESTOCK & ANIMAL HUSBANDRY", "FOOD PROCESSING & PRODUCTION", "TRADE",
        "GARMENT TEXTILE, HANDICRAFTS", "CONSTRUCTION, BUILDING AND REPAIR", "TRANSPORT & LOGISTICS",
        "BEAUTY AND WELLNESS", "OTHER"]
DISTRICTS = area1["district"].tolist()
MIN_N = 5
for c in vis.columns:
    if vis[c].dtype == object or str(vis[c].dtype) == "bool":
        if set(vis[c].dropna().unique()) <= {True, False, "True", "False"}:
            vis[c] = vis[c].map({True: 1.0, False: 0.0, "True": 1.0, "False": 0.0})
vis = vis[vis["category"].isin(CATS)].copy()
print(len(vis), "visits;", vis["category"].nunique(), "livelihoods;", vis["district"].nunique(), "districts")
FACTS = []
def add(scope, livelihood, district, statement, value, evidence, estimated, source, n, swot="none"):
    FACTS.append(dict(scope=scope, livelihood=livelihood, district=district, statement=statement,
                      value=None if value is None or (isinstance(value, float) and np.isnan(value)) else round(float(value), 4),
                      evidence_type=evidence, is_estimated=bool(estimated), source=source, n=int(n), swot=swot))

587 visits; 9 livelihoods; 5 districts


## 1. Internal facts (monitoring visits)
Rate per livelihood, Wilson 95% interval, shrinkage toward the all-livelihood average. A SWOT label is given only when the interval is clearly above or below the average.

In [6]:
GOOD = {"profit": "profit made", "saved_money": "household saved money", "coach_progressing": "coach says progressing",
        "business_running": "business running", "did_agreed_actions": "completed agreed actions"}
BAD = {"problem_sales_low": "low sales", "problem_input_cost": "high input cost", "problem_input_unavailable": "inputs unavailable",
       "problem_storage_transport": "storage or transport problem", "problem_equipment": "equipment problem",
       "problem_disaster": "disaster impact", "problem_asset_lost": "asset lost", "borrowed_money": "borrowed money",
       "skipped_meal": "skipped a meal"}
ALL = {**{k: (v, 1) for k, v in GOOD.items()}, **{k: (v, -1) for k, v in BAD.items()}}

def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h

def shrink(k, n, p0, m=10):
    return (k + m * p0) / (n + m)

def rate_table(df, keys):
    rows = []
    for key, g in df.groupby(keys):
        for col, (label, sign) in ALL.items():
            s = g[col].dropna(); n = len(s)
            if n == 0: continue
            k = s.sum(); lo, hi = wilson(k, n)
            rows.append(dict(zip(keys if isinstance(keys, list) else [keys], key if isinstance(key, tuple) else (key,)),
                              indicator=col, label=label, sign=sign, n=n, k=k, raw=k / n, lo=lo, hi=hi))
    return pd.DataFrame(rows)

base = {c: vis[c].mean() for c in ALL}
liv = rate_table(vis, ["category"])
liv["avg"] = liv["indicator"].map(base)
liv["shrunk"] = [shrink(r.k, r.n, r.avg) for r in liv.itertuples()]

def swot_internal(r):
    if r.n < MIN_N: return "none"
    if r.lo > r.avg: return "Strength" if r.sign == 1 else "Weakness"
    if r.hi < r.avg: return "Weakness" if r.sign == 1 else "Strength"
    return "none"
liv["swot"] = liv.apply(swot_internal, axis=1)

for r in liv.itertuples():
    if r.n < MIN_N: continue
    st = f"{r.category.title()}: {r.raw*100:.0f}% of visits record: {r.label} (95% CI {r.lo*100:.0f}-{r.hi*100:.0f}%, all-livelihood average {r.avg*100:.0f}%, {r.n} visits)."
    add("livelihood", r.category, None, st, r.raw * 100, "monitoring", False, "monitoring visits", r.n, r.swot)

ld = rate_table(vis[vis["district"].notna()], ["category", "district"])
ld = ld[ld["n"] >= MIN_N].copy()
ld = ld.merge(liv[["category", "indicator", "shrunk"]].rename(columns={"shrunk": "parent"}), on=["category", "indicator"])
ld["shrunk"] = [shrink(r.k, r.n, r.parent) for r in ld.itertuples()]
def swot_ld(r):
    if r.lo > r.parent: return "Strength" if r.sign == 1 else "Weakness"
    if r.hi < r.parent: return "Weakness" if r.sign == 1 else "Strength"
    return "none"
ld["swot"] = ld.apply(swot_ld, axis=1)
for r in ld.itertuples():
    st = f"{r.category.title()} in {r.district}: {r.raw*100:.0f}% of visits record: {r.label} (95% CI {r.lo*100:.0f}-{r.hi*100:.0f}%, livelihood average {r.parent*100:.0f}%, {r.n} visits)."
    add("livelihood_district", r.category, r.district, st, r.raw * 100, "monitoring", False, "monitoring visits", r.n, r.swot)

rk = vis[vis["category"].isin(CATS)].groupby("category")["record_keeping_level"].mean()
for c, v in rk.items():
    add("livelihood", c, None, f"{c.title()}: average record-keeping level is {v:.2f} on a 0-3 scale.", v, "monitoring", False, "monitoring visits", int(vis[vis.category == c]["record_keeping_level"].notna().sum()))
print(len(FACTS), "internal facts;", (liv.swot != "none").sum(), "livelihood-level labels;", (ld.swot != "none").sum(), "district-level labels")
liv[liv.swot != "none"][["category", "label", "n", "raw", "avg", "swot"]].round(2)

539 internal facts; 6 livelihood-level labels; 24 district-level labels


,category,label,n,raw,avg,swot
7,AGRICULTURE & PLANTATION CROPS,inputs unavailable,87,0.05,0.02,Weakness
13,AGRICULTURE & PLANTATION CROPS,skipped a meal,105,0.03,0.01,Weakness
59,"GARMENT TEXTILE, HANDICRAFTS",business running,83,0.88,0.94,Weakness
64,"GARMENT TEXTILE, HANDICRAFTS",storage or transport problem,71,0.04,0.12,Strength
101,TRADE,business running,62,1.00,0.94,Strength
102,TRADE,completed agreed actions,62,0.53,0.67,Weakness


## 2. Market facts and forecast check
Trend, volatility, pressure from the cleaned market features. Forecast skill is measured by a rolling backtest against a naive 'same as last month' forecast.

In [7]:
mgood = msum.copy()
for r in mgood.itertuples():
    ev = r.evidence_type; proxy = ev != "item price"
    tag = " (proxy index, not livelihood-specific)" if proxy else ""
    if r.category.startswith("ALL"): continue
    n = int(r.n_series)
    _tr = "Opportunity" if (r.median_trend_pct_per_month > 0 and r.share_rising_significant >= 0.5) else ("Threat" if r.median_trend_pct_per_month < 0 else "none")
    _weak = " (weak signal)" if _tr != "none" and not proxy else ""
    add("market", r.category, None,
        f"{r.category.title()}: median market trend is {r.median_trend_pct_per_month:+.2f}% per month across {n} {ev} series{tag}; {r.share_rising_significant*100:.0f}% are significantly rising{_weak}.",
        r.median_trend_pct_per_month, "market_" + ev.replace(" ", "_"), proxy, "market features", n, _tr if not proxy else "none")
    if pd.notna(r.median_change_12m_pct):
        add("market", r.category, None, f"{r.category.title()}: median 12-month price change is {r.median_change_12m_pct:+.1f}% ({ev}){tag}.",
            r.median_change_12m_pct, "market_" + ev.replace(" ", "_"), proxy, "market features", n)
    add("market", r.category, None, f"{r.category.title()}: market pressure is {r.pressure_label} (z={r.pressure_z:+.2f}, volatility CV {r.median_cv:.2f}) ({ev}){tag}.",
        r.pressure_z, "market_" + ev.replace(" ", "_"), proxy, "market features", n,
        "Opportunity" if (r.pressure_label == "rising" and r.enough_series) else ("Threat" if (r.pressure_label == "falling" and r.enough_series) else "none"))

ms = mmon.copy()
ms["m"] = pd.PeriodIndex(ms["month"], freq="M").astype(int)
long_keys = ms.groupby("series_key")["m"].agg(["count", "max"])
long_keys = long_keys[(long_keys["count"] >= 24) & (long_keys["max"] >= long_keys["max"].max() - 3)].index
cat_of = ms.drop_duplicates("series_key").set_index("series_key")["category"]
sample = []
for c in CATS:
    ks = [k for k in long_keys if cat_of.get(k) == c]
    if ks: sample += list(rng.choice(ks, size=min(40, len(ks)), replace=False))

def fc_drift(y, h, w=12, damp=0.5):
    d = np.diff(y[-w - 1:]).mean() if len(y) > w else np.diff(y).mean()
    return y[-1] + damp * d * h
def backtest(y, h=3, origins=6):
    e_m, e_n = [], []
    for i in range(origins):
        cut = len(y) - h - i
        if cut < 18: break
        e_m.append(abs(fc_drift(y[:cut], h) - y[cut + h - 1])); e_n.append(abs(y[cut - 1] - y[cut + h - 1]))
    return (np.mean(e_m), np.mean(e_n)) if e_m else (np.nan, np.nan)

bt = []
for k in sample:
    s = ms[ms.series_key == k].sort_values("m")
    s = s.set_index("m")["value"]
    s = s.reindex(range(s.index.min(), s.index.max() + 1)).interpolate(limit=2).dropna()
    if len(s) < 24: continue
    a, b = backtest(np.log(s.clip(lower=1e-6)).values)
    if np.isfinite(a) and np.isfinite(b) and b > 0: bt.append(dict(series_key=k, category=cat_of[k], mae_model=a, mae_naive=b, skill=1 - a / b, wins=a < b))
bt = pd.DataFrame(bt)
bt_sum = bt.groupby("category").agg(n_series=("skill", "size"), median_skill=("skill", "median"), win_rate=("wins", "mean")).reset_index()
bt_sum["forecast_usable"] = (bt_sum.win_rate >= 0.5) & (bt_sum.median_skill > 0)
for r in bt_sum.itertuples():
    ok = "beats" if r.forecast_usable else "does not beat"
    add("forecast", r.category, None, f"{r.category.title()}: price forecast {ok} the naive baseline in a rolling backtest ({r.win_rate*100:.0f}% of {r.n_series} series, median skill {r.median_skill:+.2f}); " + ("short-term forecasts can be used." if r.forecast_usable else "only the latest level and trend are reported, no forecast."),
        r.median_skill, "market_backtest", False, "rolling backtest", r.n_series)
print("overall win rate vs naive:", round(bt.wins.mean(), 3), "| median skill:", round(bt.skill.median(), 3))
bt_sum.round(3)

overall win rate vs naive: 0.444 | median skill: -0.024


,category,n_series,median_skill,win_rate,forecast_usable
0,AGRICULTURE & PLANTATION CROPS,3,-0.135,0.333,False
1,BEAUTY AND WELLNESS,1,0.088,1.000,True
2,"CONSTRUCTION, BUILDING AND REPAIR",4,-0.081,0.250,False
3,FOOD PROCESSING & PRODUCTION,3,0.173,0.667,True
4,"GARMENT TEXTILE, HANDICRAFTS",2,-0.092,0.500,False
5,LIVESTOCK & ANIMAL HUSBANDRY,1,0.045,1.000,True
6,TRANSPORT & LOGISTICS,4,-0.024,0.250,False


## 3. Area index (PCA), risk and participant facts

In [8]:
b = pd.read_excel("BRIGHT_clean_all_sheets.xlsx", sheet_name=["mod_a_household_identification", "mod_c_housing", "mod_d_assets"])
a, c, d = b["mod_a_household_identification"], b["mod_c_housing"], b["mod_d_assets"]
key = "hhcode"
h = a[[key, "a_02_district"]].merge(c[[key, "c1_05", "c1_11_1", "c1_22"]], on=key, how="left").merge(d[[key, "d_05", "d_08", "d_12", "d_15", "d_20", "d_21"]], on=key, how="left")
h["permanent_floor"] = h["c1_05"].isin([1, 2, 6]).astype(float)
h["grid_electricity"] = (h["c1_11_1"] == 1).astype(float)
h["asset_score"] = (h["d_05"].clip(upper=1).fillna(0) + (h["d_08"] > 0) + h["d_12"].fillna(0) + h["d_15"].fillna(0) + (h["d_20"] > 0) + (h["d_21"] > 0)).astype(float)
h["health_minutes"] = pd.to_numeric(h["c1_22"], errors="coerce").clip(upper=180)
dist_names = dict(zip(range(1, 26), DISTRICTS))
h["district"] = h["a_02_district"].map(dist_names)
agg = h.groupby("district").agg(permanent_floor=("permanent_floor", "mean"), grid_electricity=("grid_electricity", "mean"),
                                asset_score=("asset_score", "mean"), health_minutes=("health_minutes", "mean"), households=(key, "size")).reset_index()
A = area1.merge(agg, on="district")
FEATS = [c for c in A.columns if c not in ("district", "households", "grid_electricity")]
X = StandardScaler().fit_transform(A[FEATS])
pca = PCA(n_components=min(3, len(FEATS))).fit(X)
sc = pca.transform(X)[:, 0]
load = pd.Series(pca.components_[0], index=FEATS)
good_dir = np.mean([load.get("permanent_floor", 0), load.get("grid_electricity", 0), load.get("asset_score", 0), -load.get("health_minutes", 0)])
if good_dir < 0: sc, load = -sc, -load
A["area_index"] = (100 * (sc - sc.min()) / (sc.max() - sc.min())).round(1)
A["area_rank"] = A["area_index"].rank(ascending=False, method="min").astype(int)
print("PC1 explained variance:", round(pca.explained_variance_ratio_[0], 3), "| PC1+PC2:", round(pca.explained_variance_ratio_[:2].sum(), 3))
print(load.round(2).to_dict())

for r in A.itertuples():
    _q75, _q25, _med = A.area_index.quantile(0.75), A.area_index.quantile(0.25), A.area_index.median()
    _ar = "Opportunity" if r.area_index >= _q75 else ("Threat" if r.area_index <= _q25 else ("Opportunity" if r.area_index >= _med else "Threat"))
    _aw = "" if (r.area_index >= _q75 or r.area_index <= _q25) else " (weak signal)"
    add("area", None, r.district, f"{r.district}: area development index {r.area_index:.0f}/100 (rank {r.area_rank} of 25; higher = better-off){_aw}; floor {r.permanent_floor*100:.0f}% permanent, grid electricity {r.grid_electricity*100:.0f}%, {r.health_minutes:.0f} min to nearest health facility.",
        r.area_index, "bright_survey", False, "BRIGHT survey (PCA)", r.households,
        _ar)

dis["month"] = pd.PeriodIndex(dis["month"], freq="M")
last12 = dis[dis["month"] >= dis["month"].max() - 11]
risk = last12.groupby("district")["affected_families"].sum().reindex(DISTRICTS)
risk_known = risk.dropna()
risk_pct = risk.rank(pct=True) * 100
for dname, v in risk.items():
    if pd.isna(v):
        add("risk", None, dname, f"{dname}: no disaster records available in the disaster table (unknown, not zero risk).", None, "disaster_records", True, "disaster table", 0); continue
    add("risk", None, dname, f"{dname}: {int(v)} families affected by disasters in the latest 12 months (higher than {risk_pct[dname]:.0f}% of districts with records){'' if risk_pct[dname] >= 75 else ' (weak signal)'}.",
        v, "disaster_records", False, "disaster table", int(v), "Threat" if risk_pct[dname] >= 50 else "Opportunity")

p = people[people["category"].isin(CATS)]
for c_, g in p.groupby("category"):
    fem = (g["gender"] == "Female").mean() * 100
    add("people", c_, None, f"{c_.title()}: {len(g)} participants, {fem:.0f}% female, mean age {g['age'].mean():.0f}.", len(g), "participant_profile", False, "participant records", len(g))
print(len(FACTS), "facts so far")
A[["district", "area_index", "area_rank", "households"]].sort_values("area_rank").head(25)

PC1 explained variance: 0.406 | PC1+PC2: 0.653
{'food_insecurity_score': -0.42, 'debt_pct': 0.05, 'land_pct': 0.48, 'permanent_floor': 0.19, 'asset_score': 0.61, 'health_minutes': -0.42}
646 facts so far


,district,area_index,area_rank,households
4,Hambantota,100.0,1,205
14,Anuradhapura,91.2,2,306
20,Puttalam,85.4,3,246
2,Kalutara,82.4,4,375
19,Kurunegala,80.2,5,506
5,Matara,76.9,6,263
1,Gampaha,72.6,7,659
10,Vavuniya,69.8,8,57
15,Polonnaruwa,69.4,9,122
0,Colombo,64.8,10,715


## 4. Viability Index
Four sub-scores (0-100), equal weights (25% each) as the base, entropy weights kept as a sensitivity check, then 1,000 random weight perturbations to give a score range and rank stability.

In [9]:
def pct(s): return s.rank(pct=True) * 100
cat_market = msum[~msum.category.str.startswith("ALL")].copy()
cat_market["w"] = np.where(cat_market.evidence_type == "item price", 1.0, 0.5)
mk = {}
for c_ in CATS:
    g = cat_market[cat_market.category == c_]
    if len(g) == 0: mk[c_] = dict(trend=np.nan, cv=np.nan, press=np.nan); continue
    w = g.w.values / g.w.sum()
    mk[c_] = dict(trend=(g.median_trend_pct_per_month * w).sum(), cv=(g.median_cv * w).sum(), press=(g.pressure_z * w).sum())
mk = pd.DataFrame(mk).T
mk["market"] = (pct(mk.trend) + (100 - pct(mk.cv)) + pct(mk.press)) / 3
cnt = cat_market.assign(item=(cat_market.evidence_type == "item price") * cat_market.n_series, prox=(cat_market.evidence_type != "item price") * cat_market.n_series).groupby("category")[["item", "prox"]].sum()
mk["conf"] = ((cnt["item"] + 0.3 * cnt["prox"]) / 10).clip(upper=1).reindex(mk.index).fillna(0)
mk["market"] = (50 + mk["conf"] * (mk["market"].fillna(50) - 50))

prof_cols = ["profit", "saved_money", "coach_progressing"]
res_cols = ["problem_disaster", "skipped_meal", "problem_input_unavailable"]
def sub(df, cols, parent=None):
    out = {}
    for col in cols:
        s = df[col].dropna(); out[col] = (s.sum(), len(s))
    return out
def score_group(df, cols, invert=False, parent_rate=None):
    vals = []
    for col in cols:
        s = df[col].dropna(); p0 = base[col]
        pr = parent_rate[col] if parent_rate is not None else p0
        vals.append(shrink(s.sum(), len(s), pr))
    m = np.mean(vals) * 100
    return 100 - m if invert else m

rows = []
risk_s = (100 - risk.rank(pct=True) * 100).fillna(50)
for c_ in CATS:
    gl = vis[vis.category == c_]
    p_l = {col: liv[(liv.category == c_) & (liv.indicator == col)].shrunk.values[0] if len(liv[(liv.category == c_) & (liv.indicator == col)]) else base[col] for col in prof_cols + res_cols}
    for dn in DISTRICTS:
        gd = gl[gl.district == dn]
        use_d = len(gd) >= MIN_N
        prof = score_group(gd, prof_cols, parent_rate=p_l) if use_d else score_group(gl, prof_cols)
        res_in = score_group(gd, res_cols, invert=True, parent_rate=p_l) if use_d else score_group(gl, res_cols, invert=True)
        resil = 0.5 * res_in + 0.5 * risk_s[dn]
        rows.append(dict(livelihood=c_, district=dn, profitability=prof, market=mk.loc[c_, "market"],
                         area=A.set_index("district").loc[dn, "area_index"], resilience=resil, district_evidence=use_d,
                         n_livelihood_visits=len(gl), n_district_visits=len(gd)))
S = pd.DataFrame(rows)
SUB = ["profitability", "market", "area", "resilience"]

M = S[SUB].values / 100 + 1e-9
P = M / M.sum(0)
E = -(P * np.log(P)).sum(0) / np.log(len(M))
w_ent = (1 - E) / (1 - E).sum()
w0 = np.ones(4) / 4
print("entropy weights (sensitivity only):", {k: round(float(v), 3) for k, v in zip(SUB, w_ent)})
S["viability"] = S[SUB].values @ w0

W = rng.dirichlet(w0 * 60, size=1000)
sims = S[SUB].values @ W.T
S["score_p5"] = np.percentile(sims, 5, axis=1); S["score_p95"] = np.percentile(sims, 95, axis=1)
S["score_range"] = S["score_p95"] - S["score_p5"]
base_rank = S.groupby("district")["viability"].rank(ascending=False, method="first").values
stab = np.zeros(len(S))
for j in range(1000):
    tmp = S.assign(_s=sims[:, j]); r = tmp.groupby("district")["_s"].rank(ascending=False, method="first").values
    stab += (r == base_rank)
S["rank_stability"] = stab / 1000
S["rank_in_district"] = base_rank.astype(int)
S["viability"] = S.viability.round(1); S[["score_p5", "score_p95", "score_range"]] = S[["score_p5", "score_p95", "score_range"]].round(1); S["rank_stability"] = S.rank_stability.round(3)

for r in S.itertuples():
    add("viability", r.livelihood, r.district,
        f"{r.livelihood.title()} in {r.district}: viability {r.viability:.0f}/100 (range {r.score_p5:.0f}-{r.score_p95:.0f}); profitability {r.profitability:.0f}, market {r.market:.0f}, area {r.area:.0f}, resilience {r.resilience:.0f}; rank {r.rank_in_district} of 9 in this district.",
        r.viability, "computed_score", not r.district_evidence, "viability index", r.n_district_visits if r.district_evidence else r.n_livelihood_visits)
S.sort_values("viability", ascending=False).head(10).round(1)

entropy weights (sensitivity only): {'profitability': 0.02, 'market': 0.095, 'area': 0.773, 'resilience': 0.112}


,livelihood,district,profitability,market,area,resilience,district_evidence,n_livelihood_visits,n_district_visits,viability,score_p5,score_p95,score_range,rank_stability,rank_in_district
79,TRADE,Hambantota,58.9,63.0,100.0,68.5,False,62,0,72.6,69.5,76.2,6.7,1.0,1
94,TRADE,Kurunegala,58.9,63.0,80.2,86.0,False,62,0,72.0,69.6,74.3,4.6,1.0,1
77,TRADE,Kalutara,58.9,63.0,82.4,81.0,False,62,0,71.3,69.1,73.5,4.3,1.0,1
89,TRADE,Anuradhapura,58.9,63.0,91.2,71.0,False,62,0,71.0,68.6,73.8,5.3,1.0,1
85,TRADE,Vavuniya,58.9,63.0,69.8,91.0,False,62,0,70.7,68.1,73.2,5.1,1.0,1
54,FOOD PROCESSING & PRODUCTION,Hambantota,48.5,61.1,100.0,67.2,False,55,0,69.2,65.3,73.5,8.1,0.7,2
129,"CONSTRUCTION, BUILDING AND REPAIR",Hambantota,51.4,56.1,100.0,68.4,False,59,0,69.0,65.2,73.3,8.1,0.4,4
204,OTHER,Hambantota,49.4,59.3,100.0,67.4,False,62,0,69.0,65.2,73.3,8.1,0.6,3
29,LIVESTOCK & ANIMAL HUSBANDRY,Hambantota,50.4,57.4,100.0,67.7,False,59,0,68.9,65.2,73.2,8.0,0.6,5
69,FOOD PROCESSING & PRODUCTION,Kurunegala,48.5,61.1,80.2,84.7,False,55,0,68.6,65.6,71.5,5.9,0.7,2


## 5. Evaluation

In [10]:
EV = []
def ev(check, metric, value, note=""): EV.append(dict(check=check, metric=metric, value=value, note=note))

# shrinkage: split-half prediction
idx = np.arange(len(vis)); r1 = np.random.default_rng(1)
errs_raw, errs_shr = [], []
for rep in range(200):
    msk = r1.random(len(vis)) < 0.5
    A_, B_ = vis[msk], vis[~msk]
    for col in prof_cols:
        pb = B_.groupby("category")[col].mean()
        gm = A_[col].mean()
        for c_, g in A_.groupby("category"):
            s = g[col].dropna()
            if len(s) == 0 or c_ not in pb.index or np.isnan(pb[c_]): continue
            errs_raw.append((s.mean() - pb[c_]) ** 2); errs_shr.append((shrink(s.sum(), len(s), gm) - pb[c_]) ** 2)
ev("shrinkage", "split-half MSE raw rate", np.mean(errs_raw)); ev("shrinkage", "split-half MSE shrunk rate", np.mean(errs_shr),
   "lower is better")
ev("shrinkage", "MSE reduction %", 100 * (1 - np.mean(errs_shr) / np.mean(errs_raw)))

ev("forecast", "series backtested", len(bt)); ev("forecast", "win rate vs naive", bt.wins.mean(), "share of series where drift beats last value")
ev("forecast", "median skill", bt.skill.median(), "1 - MAE_model / MAE_naive")
ev("area_pca", "PC1 explained variance", pca.explained_variance_ratio_[0]); ev("area_pca", "PC1+PC2 explained variance", pca.explained_variance_ratio_[:2].sum())
for k_, v_ in load.items(): ev("area_pca", f"PC1 loading {k_}", v_)
ev("viability", "mean score range", S.score_range.mean(), "5th-95th percentile width over 1000 weight draws")
ev("viability", "mean rank stability", S.rank_stability.mean(), "share of draws keeping the same rank in district")
for k_, v_ in zip(SUB, w_ent): ev("viability", f"entropy weight {k_} (sensitivity; base is 0.25 each)", v_)
alt = S[SUB].values @ w_ent
rho = []
for dn in DISTRICTS:
    m_ = (S.district == dn).values
    rho.append(stats.spearmanr(S.loc[m_, "viability"], alt[m_])[0])
ev("viability", "mean Spearman rank agreement, equal vs entropy weights", np.nanmean(rho), "within-district livelihood ranking")
ev("labels", "SWOT-labelled facts", sum(f["swot"] != "none" for f in FACTS), "of " + str(len(FACTS)))
ev("coverage", "livelihood x district pairs with district evidence", int(S.district_evidence.sum()), "of " + str(len(S)))
EV = pd.DataFrame(EV); EV["value"] = EV["value"].astype(float).round(4)
EV

,check,metric,value,note
0,shrinkage,split-half MSE raw rate,0.0133,
1,shrinkage,split-half MSE shrunk rate,0.0106,lower is better
2,shrinkage,MSE reduction %,20.1621,
3,forecast,series backtested,18.0000,
4,forecast,win rate vs naive,0.4444,share of series where drift beats last value
5,forecast,median skill,-0.0244,1 - MAE_model / MAE_naive
6,area_pca,PC1 explained variance,0.4060,
7,area_pca,PC1+PC2 explained variance,0.6530,
8,area_pca,PC1 loading food_insecurity_score,-0.4213,
9,area_pca,PC1 loading debt_pct,0.0461,


## 6. Charts and save

In [11]:
import matplotlib; import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 5))
t = S.groupby("livelihood")["viability"].mean().sort_values()
ax.barh([x.title()[:32] for x in t.index], t.values, color="#3b6ea5"); ax.set_xlabel("Mean viability score"); plt.tight_layout()
fig.savefig(OUT / "viability_by_livelihood.png", dpi=150); plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 5))
q = A.sort_values("area_index"); ax.barh(q.district, q.area_index, color="#4a8f6a"); ax.set_xlabel("Area index (PCA)"); plt.tight_layout()
fig.savefig(OUT / "area_index_pca.png", dpi=150); plt.close(fig)

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(bt.skill.clip(-1, 1), bins=30, color="#8a6bbf"); ax.axvline(0, color="k", lw=1); ax.set_xlabel("Forecast skill vs naive"); plt.tight_layout()
fig.savefig(OUT / "forecast_skill.png", dpi=150); plt.close(fig)

fb = pd.DataFrame(FACTS); fb.insert(0, "fact_id", [f"F{i:04d}" for i in range(1, len(fb) + 1)])
fb.to_csv(OUT / "fact_bank.csv", index=False)
S.to_csv(OUT / "livelihood_scores.csv", index=False)
A.to_csv(OUT / "area_index_pca.csv", index=False)
EV.to_csv(OUT / "evaluation_results.csv", index=False)
bt_sum.to_csv(OUT / "forecast_backtest.csv", index=False)
import shutil; shutil.make_archive("model2_outputs", "zip", OUT)
print(fb.scope.value_counts().to_dict(), "| SWOT labels:", fb.swot.value_counts().to_dict())
print(sorted(os.listdir(OUT)))
try:
    from google.colab import files; files.download("model2_outputs.zip")
except Exception: pass

{'livelihood_district': 404, 'viability': 225, 'livelihood': 135, 'market': 41, 'risk': 25, 'area': 25, 'people': 9, 'forecast': 7} | SWOT labels: {'none': 789, 'Opportunity': 27, 'Threat': 25, 'Weakness': 19, 'Strength': 11}
['area_index_pca.csv', 'area_index_pca.png', 'evaluation_results.csv', 'fact_bank.csv', 'forecast_backtest.csv', 'forecast_skill.png', 'livelihood_scores.csv', 'viability_by_livelihood.png']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>